In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              mean_absolute_error, r2_score)
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Set all random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Random seed: {SEED} (results will be identical every run)")

Using device: cpu
Random seed: 42 (results will be identical every run)


In [15]:
df = pd.read_csv("../data/processed/features.csv")
df = df.sort_values(['Email', 'assignment_order']).reset_index(drop=True)

students = df['Email'].unique()
np.random.seed(42)
np.random.shuffle(students)
split_idx = int(len(students) * 0.8)
train_students = students[:split_idx]
test_students  = students[split_idx:]

df_train = df[df['Email'].isin(train_students)].copy()
df_test  = df[df['Email'].isin(test_students)].copy()

FEATURE_COLS = [
    'grade_pct', 'time_minutes', 'never_started',
    'grade_zscore', 'time_zscore', 'effort_efficiency',
    'class_skip_rate', 'grade_velocity', 'time_velocity',
    'rolling_avg_grade_3', 'cumulative_skips',
    'cumulative_avg_grade', 'difficulty_score'
]

scaler = StandardScaler()
df_train[FEATURE_COLS] = scaler.fit_transform(df_train[FEATURE_COLS])
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])

print(f"Train: {len(train_students)} students | Test: {len(test_students)} students")
print(f"Features: {len(FEATURE_COLS)}")

Train: 33 students | Test: 9 students
Features: 13


In [16]:
class StudentSequenceDataset(Dataset):
    def __init__(self, dataframe, feature_cols):
        self.sequences = []
        self.grade_targets = []
        self.risk_labels = []

        for student_email, group in dataframe.groupby('Email'):
            group = group.sort_values('assignment_order')
            features = group[feature_cols].values.astype(np.float32)
            grade_targets = (group['next_grade'].values / 100.0).astype(np.float32)
            risk_label = float(group['at_risk'].iloc[0])

            self.sequences.append(torch.tensor(features))
            self.grade_targets.append(torch.tensor(grade_targets))
            self.risk_labels.append(torch.tensor(risk_label))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (self.sequences[idx],
                self.grade_targets[idx],
                self.risk_labels[idx])

train_dataset = StudentSequenceDataset(df_train, FEATURE_COLS)
test_dataset  = StudentSequenceDataset(df_test,  FEATURE_COLS)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Datasets created: {len(train_dataset)} train, {len(test_dataset)} test")

Datasets created: 33 train, 9 test


In [17]:
class AcademicCNNGRU(nn.Module):
    """
    CNN-GRU: Conv1D extracts local patterns, GRU models temporal sequence.

    WHY add CNN before GRU:
    - GRU alone processes one step at a time and learns long-range dependencies
    - CNN slides a kernel across consecutive assignments to detect LOCAL patterns
    - Example: kernel_size=3 means "look at 3 consecutive assignments at once"
    - It can detect: 3 consecutive fails, sudden grade spike, early dropout pattern
    - The CNN output (filtered features) then feeds into GRU as enriched input

    Architecture:
    Input [batch, 19, 13]
      ↓ (transpose for Conv1D which expects [batch, channels, length])
    [batch, 13, 19]
      ↓
    Conv1D (kernel=3) → local patterns extracted
    [batch, 64, 19]
      ↓ (transpose back)
    [batch, 19, 64]
      ↓
    GRU → temporal modeling
    [batch, 19, hidden_size]
      ↓
    Attention → context vector
      ↓
    Grade head + Risk head
    """

    def __init__(self, input_size=13, cnn_channels=64,
                 hidden_size=64, num_layers=2, dropout=0.3):
        super(AcademicCNNGRU, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Conv1D layer
        # in_channels = input features (13)
        # out_channels = number of filters (64) — like having 64 pattern detectors
        # kernel_size = 3 means each filter looks at 3 consecutive assignments
        # padding=1 keeps sequence length the same (19 → 19)
        self.conv1d = nn.Conv1d(
            in_channels=input_size,
            out_channels=cnn_channels,
            kernel_size=3,
            padding=1
        )
        self.conv_bn  = nn.BatchNorm1d(cnn_channels)  # stabilizes conv output
        self.conv_act = nn.ReLU()
        self.conv_drop = nn.Dropout(dropout)

        # GRU takes CNN output (cnn_channels) as input, not original features
        self.gru = nn.GRU(
            input_size=cnn_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout   = nn.Dropout(dropout)
        self.attention = nn.Linear(hidden_size, 1)

        self.grade_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

        self.risk_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [batch, seq=19, features=13]

        # Conv1D expects [batch, channels, length]
        x_conv = x.transpose(1, 2)          # [batch, 13, 19]
        x_conv = self.conv1d(x_conv)         # [batch, 64, 19]
        x_conv = self.conv_bn(x_conv)
        x_conv = self.conv_act(x_conv)
        x_conv = self.conv_drop(x_conv)
        x_conv = x_conv.transpose(1, 2)     # [batch, 19, 64] — back to sequence format

        # GRU
        batch_size = x_conv.size(0)
        h0 = torch.zeros(self.num_layers, batch_size,
                         self.hidden_size).to(x.device)
        gru_out, _ = self.gru(x_conv, h0)  # [batch, 19, hidden]
        gru_out = self.dropout(gru_out)

        # Attention
        attn_scores  = self.attention(gru_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (attn_weights * gru_out).sum(dim=1)

        # Heads
        grade_pred = self.grade_head(gru_out).squeeze(-1)   # [batch, 19]
        risk_pred  = self.risk_head(context).squeeze(-1)    # [batch]

        return grade_pred, risk_pred, attn_weights.squeeze(-1)


model_v2 = AcademicCNNGRU(
    input_size=13,
    cnn_channels=64,
    hidden_size=64,
    num_layers=2,
    dropout=0.3
).to(device)

total_params = sum(p.numel() for p in model_v2.parameters())
print(f"CNN-GRU architecture:")
print(model_v2)
print(f"\nTotal parameters: {total_params:,}")
print(f"\nGRU v1 parameters: 44,419")
print(f"CNN-GRU v2 parameters: {total_params:,}")
print(f"Additional params from CNN: {total_params - 44419:,}")

CNN-GRU architecture:
AcademicCNNGRU(
  (conv1d): Conv1d(13, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_act): ReLU()
  (conv_drop): Dropout(p=0.3, inplace=False)
  (gru): GRU(64, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (attention): Linear(in_features=64, out_features=1, bias=True)
  (grade_head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
  (risk_head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

Total parameters: 56,899

GRU v1 parameters: 44,419
CNN-GRU v2 parameters: 56,899
Additional params from CNN: 12,480


In [18]:
mlflow.end_run()

optimizer_v2 = torch.optim.AdamW(
    model_v2.parameters(), lr=0.001, weight_decay=1e-4
)
scheduler_v2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v2, mode='min', patience=10, factor=0.5
)
criterion_grade = nn.MSELoss()
criterion_risk  = nn.BCELoss()

N_EPOCHS = 150
ALPHA, BETA = 0.5, 0.5

train_losses_v2  = []
val_losses_v2    = []
best_val_loss_v2    = float('inf')
best_model_state_v2 = None

mlflow.set_experiment("academic_performance_gru")

with mlflow.start_run(run_name="cnn_gru_v2_multitask"):

    mlflow.log_params({
        "model": "CNN_GRU_v2",
        "cnn_channels": 64,
        "kernel_size": 3,
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "epochs": N_EPOCHS,
        "lr": 0.001,
    })

    for epoch in range(N_EPOCHS):

        model_v2.train()
        train_loss = 0.0
        for features, grade_targets, risk_labels in train_loader:
            features      = features.to(device)
            grade_targets = grade_targets.to(device)
            risk_labels   = risk_labels.to(device).float()

            optimizer_v2.zero_grad()
            grade_pred, risk_pred, _ = model_v2(features)

            loss = (ALPHA * criterion_grade(grade_pred, grade_targets) +
                    BETA  * criterion_risk(risk_pred, risk_labels))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_v2.parameters(), max_norm=1.0)
            optimizer_v2.step()
            train_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        train_losses_v2.append(avg_train)

        model_v2.eval()
        val_loss = 0.0
        with torch.no_grad():
            for features, grade_targets, risk_labels in test_loader:
                features      = features.to(device)
                grade_targets = grade_targets.to(device)
                risk_labels   = risk_labels.to(device).float()

                grade_pred, risk_pred, _ = model_v2(features)
                loss = (ALPHA * criterion_grade(grade_pred, grade_targets) +
                        BETA  * criterion_risk(risk_pred, risk_labels))
                val_loss += loss.item()

        avg_val = val_loss / len(test_loader)
        val_losses_v2.append(avg_val)
        scheduler_v2.step(avg_val)

        if avg_val < best_val_loss_v2:
            best_val_loss_v2    = avg_val
            best_model_state_v2 = model_v2.state_dict().copy()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | "
                  f"Train: {avg_train:.4f} | "
                  f"Val: {avg_val:.4f} | "
                  f"LR: {optimizer_v2.param_groups[0]['lr']:.6f}")

    print(f"\nBest val loss: {best_val_loss_v2:.4f}")
    mlflow.log_metric("best_val_loss", best_val_loss_v2)

Epoch  10/150 | Train: 0.4369 | Val: 0.3937 | LR: 0.001000
Epoch  20/150 | Train: 0.3233 | Val: 0.3214 | LR: 0.001000
Epoch  30/150 | Train: 0.4054 | Val: 0.2288 | LR: 0.001000
Epoch  40/150 | Train: 0.1767 | Val: 0.1489 | LR: 0.001000
Epoch  50/150 | Train: 0.0990 | Val: 0.1306 | LR: 0.001000
Epoch  60/150 | Train: 0.1169 | Val: 0.0883 | LR: 0.001000
Epoch  70/150 | Train: 0.0624 | Val: 0.2674 | LR: 0.001000
Epoch  80/150 | Train: 0.0439 | Val: 0.1439 | LR: 0.000500
Epoch  90/150 | Train: 0.0257 | Val: 0.1421 | LR: 0.000250
Epoch 100/150 | Train: 0.0261 | Val: 0.1370 | LR: 0.000125
Epoch 110/150 | Train: 0.0384 | Val: 0.1687 | LR: 0.000125
Epoch 120/150 | Train: 0.0318 | Val: 0.1098 | LR: 0.000063
Epoch 130/150 | Train: 0.0268 | Val: 0.1331 | LR: 0.000031
Epoch 140/150 | Train: 0.0242 | Val: 0.2131 | LR: 0.000016
Epoch 150/150 | Train: 0.0589 | Val: 0.1542 | LR: 0.000008

Best val loss: 0.0641


In [20]:
model_v2.load_state_dict(best_model_state_v2)
model_v2.eval()

all_grade_preds = []
all_grade_true  = []
all_risk_preds  = []
all_risk_true   = []

with torch.no_grad():
    for features, grade_targets, risk_labels in test_loader:
        features = features.to(device)
        grade_pred, risk_pred, _ = model_v2(features)

        all_grade_preds.extend((grade_pred.cpu().numpy() * 100).flatten())
        all_grade_true.extend((grade_targets.numpy() * 100).flatten())

        risk_binary = (risk_pred.cpu().numpy() > 0.5).astype(int)
        all_risk_preds.extend(risk_binary)
        all_risk_true.extend(risk_labels.numpy().astype(int))

grade_mae_v2 = mean_absolute_error(all_grade_true, all_grade_preds)
grade_r2_v2  = r2_score(all_grade_true, all_grade_preds)
risk_acc_v2  = accuracy_score(all_risk_true, all_risk_preds)

mlflow.log_metrics({
    "cnn_gru_grade_mae": grade_mae_v2,
    "cnn_gru_grade_r2":  grade_r2_v2,
    "cnn_gru_risk_accuracy": risk_acc_v2
})

print("=" * 60)
print("FINAL COMPARISON — All Models")
print("=" * 60)
print()
print("CLASSIFICATION ACCURACY (at-risk prediction):")
print(f"  Logistic Regression:  88.3%")
print(f"  Random Forest:        86.5%")
print(f"  GRU v1:               88.9%")
print(f"  CNN-GRU v2:           {risk_acc_v2*100:.1f}%")
print()
print("REGRESSION (next assignment grade):")
print(f"  Random Forest MAE:    15.51 pts  R²: 0.750")
print(f"  GRU v1 MAE:           22.18 pts  R²: 0.197")
print(f"  CNN-GRU v2 MAE:       {grade_mae_v2:.2f} pts  R²: {grade_r2_v2:.3f}")
print()
print("Detailed CNN-GRU classification:")
print(classification_report(all_risk_true, all_risk_preds,
                            target_names=['Not At Risk', 'At Risk']))

import os
os.makedirs("../outputs/models", exist_ok=True)
torch.save({
    'model_state_dict': best_model_state_v2,
    'model_config': {
        'input_size': 13, 'cnn_channels': 64,
        'hidden_size': 64, 'num_layers': 2, 'dropout': 0.3
    },
    'scaler': scaler,
    'feature_cols': FEATURE_COLS,
    'metrics': {
        'grade_mae': grade_mae_v2,
        'grade_r2': grade_r2_v2,
        'risk_accuracy': risk_acc_v2
    }
}, "../outputs/models/cnn_gru_v2.pth")

print("\nModel saved: outputs/models/cnn_gru_v2.pth")
print("Phase 5 complete.")

FINAL COMPARISON — All Models

CLASSIFICATION ACCURACY (at-risk prediction):
  Logistic Regression:  88.3%
  Random Forest:        86.5%
  GRU v1:               88.9%
  CNN-GRU v2:           77.8%

REGRESSION (next assignment grade):
  Random Forest MAE:    15.51 pts  R²: 0.750
  GRU v1 MAE:           22.18 pts  R²: 0.197
  CNN-GRU v2 MAE:       7.40 pts  R²: 0.853

Detailed CNN-GRU classification:
              precision    recall  f1-score   support

 Not At Risk       0.75      1.00      0.86         6
     At Risk       1.00      0.33      0.50         3

    accuracy                           0.78         9
   macro avg       0.88      0.67      0.68         9
weighted avg       0.83      0.78      0.74         9


Model saved: outputs/models/cnn_gru_v2.pth
Phase 5 complete.
